## LangChain之内存记忆

大多数LLM应用都具有对话功能，如聊天机器人，记住先前的交互非常关键。对话的重要一环是能够引用之前提及的信息，这些信息需要进行存储，因此将这种存储过去交互信息的能力称为记忆 ( Memory )。

**没有Memory**

In [1]:
from langchain.agents import create_agent
from dotenv import load_dotenv
load_dotenv()
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model

model = init_chat_model(model="deepseek:deepseek-chat")

agent = create_agent(model=model)

# 第一轮
agent.invoke({"messages": [{"role": "user", "content": "我叫张三"}]})

# 第二轮 - 不记得第一轮！
response = agent.invoke({"messages": [{"role": "user", "content": "我叫什么？"}]})
# AI 会说"不知道"
for message in response["messages"]:
    message.pretty_print()

D:\python_virtualenv\langchianv1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


================================ Human Message =================================

我叫什么？
================================== Ai Message ==================================

我暂时还不知道你的名字呢！😊 你可以告诉我你的名字，这样我就能记住并且用名字称呼你了。或者如果你有特定的昵称或代号，我也可以使用那个来称呼你。

有什么我可以帮助你的吗？无论你想聊什么话题，或者需要什么帮助，我都很乐意为你提供支持！✨


**添加Memory**

In [2]:
from langgraph.checkpoint.memory import InMemorySaver

# 1. 创建 Agent 时添加 checkpointer
agent = create_agent(
    model=model,
    tools=[],
    checkpointer=InMemorySaver()  # 添加Memory
)

# 2. 调用时指定 thread_id
config = {"configurable": {"thread_id": "conversation_1"}}

# 第一轮
agent.invoke(
    {"messages": [{"role": "user", "content": "我叫张三"}]},
    config=config
)

# 第二轮 - 记得第一轮！
response = agent.invoke(
    {"messages": [{"role": "user", "content": "我叫什么？"}]},
    config=config
)
# AI 会说"你叫张三"
for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

我叫张三
================================== Ai Message ==================================

你好张三！很高兴认识你！😊

我是DeepSeek，由深度求索公司创造的AI助手。有什么我可以帮助你的吗？无论是回答问题、协助思考问题、还是日常聊天，我都很乐意为你提供帮助！

你今天过得怎么样？有什么特别想聊的话题或者需要我协助的事情吗？
================================ Human Message =================================

我叫什么？
================================== Ai Message ==================================

你刚才告诉我，你叫**张三**！😊  

如果这是个测试或者你想让我用其他名字称呼你，随时告诉我哦～


**Memory + 工具**

In [4]:
@tool
def search(query: str) -> str:
    """搜索工具"""
    return f"关于 {query} 的结果..."
    

agent = create_agent(
    model=model,
    tools=[search],
    checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id": "session_1"}}

# 第一轮：使用工具
agent.invoke({"messages": [{"role": "user", "content": "搜索 Python"}]}, config)
# Agent 调用 search("Python")

# 第二轮：引用之前的结果
response = agent.invoke(
    {"messages": [{"role": "user", "content": "刚才搜索的结果是什么？"}]},
    config
)
# Agent 记得工具返回的结果，无需重新调用
for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

刚才搜索的结果是什么？
================================== Ai Message ==================================

我刚才没有执行任何搜索操作。如果您想了解某个特定主题的搜索结果，请告诉我您想搜索什么内容，我可以帮您进行搜索查询。

您是想了解某个特定话题的信息吗？


**客服系统实战**

In [6]:
# 1. 基础环境
from dotenv import load_dotenv
load_dotenv()

from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model

# 2. 定义工具（Tools）
@tool
def query_order_status(order_id: str) -> str:
    """根据订单号查询订单状态"""
    return f"订单 {order_id} 当前状态：已支付，待发货"


@tool
def query_shipping_status(order_id: str) -> str:
    """根据订单号查询物流信息"""
    return f"订单 {order_id} 的物流信息：顺丰快递，运输中"


# 3. 初始化模型
model = init_chat_model(model="deepseek:deepseek-chat")


# 4. 初始化 Checkpointer（记忆）
checkpointer = InMemorySaver()


# 5. 创建 Agent
agent = create_agent(
    model=model,
    tools=[query_order_status, query_shipping_status],
    system_prompt=(
        "你是一个电商客服助手。"
        "如果用户在对话中提供了订单号，你需要记住它。"
        "当用户询问订单状态或物流信息时，如果已经知道订单号，"
        "请直接使用工具查询；如果不知道订单号，请先向用户确认。"
    ),
    checkpointer=checkpointer
)


# 6. 对外服务函数
def customer_service(session_id, message):
    """
    session_id: 用户会话 ID（同一个 ID 共享记忆）
    message: 用户输入
    """
    config = {
        "configurable": {
            "thread_id": session_id
        }
    }

    result = agent.invoke(
        {
            "messages": [
                {"role": "user", "content": message}
            ]
        },
        config=config
    )

    return result["messages"][-1].content


# 7. 示例运行
if __name__ == "__main__":
    print(customer_service("user_001", "你好"))
    print("="*50)
    print(customer_service("user_002", "我的订单号是12345"))
    print("="*50)
    print(customer_service("user_001", "我的订单号是123456"))
    print("="*50)
    print(customer_service("user_002", "帮我查一下物流"))

你好！我是电商客服助手，很高兴为您服务！

我可以帮助您：
1. 查询订单状态
2. 查询物流信息

如果您需要查询订单相关信息，请提供您的订单号，我会帮您查询最新状态。
好的，我已经记下了您的订单号：12345。

请问您需要查询什么信息呢？我可以帮您查询：
1. 订单状态
2. 物流信息

请告诉我您想了解哪方面的信息？
好的，我已经记下了您的订单号：123456。

请问您需要查询什么信息呢？
- 订单状态
- 物流信息
- 或者其他关于这个订单的问题

请告诉我您需要什么帮助。
根据查询结果，您的订单12345的物流信息如下：

**快递公司**：顺丰快递
**物流状态**：运输中

您的包裹目前正在运输途中，请耐心等待。如果您需要更详细的物流跟踪信息，建议您直接访问顺丰快递官网或使用顺丰快递APP，输入快递单号进行实时跟踪。


**SqliteSaver**

In [9]:
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()


model = init_chat_model(
    model="deepseek:deepseek-chat",
    temperature=0.1,
    max_tokens=2000
)

# 创建持久化 checkpointer（使用 with 语句）
# with SqliteSaver.from_conn_string("D:/GP/sqdata/checkpoints.sqlite") as checkpointer:
#     agent = create_agent(
#         model=model,
#         tools=[],
#         system_prompt="你是一个有帮助的助手。",
#         checkpointer=checkpointer  # 使用 SQLite
#     )
#     config = {"configurable": {"thread_id": "user_123"}}
#     # 第一次运行
#     agent.invoke({"messages": [{"role": "user", "content": "我的订单号是12345"}]}, config)

config = {"configurable": {"thread_id": "user_123"}}
# 程序重启后，对话仍然保留！
with SqliteSaver.from_conn_string("D:/GP/sqdata/checkpoints.sqlite") as checkpointer:
    agent = create_agent(model=model, checkpointer=checkpointer,system_prompt="你是一个有帮助的助手。",)
    response = agent.invoke({"messages": [{"role": "user", "content": "我的订单号是多少？"}]}, config)

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

我的订单号是12345
================================== Ai Message ==================================

您好！您的订单号 **12345** 已收到。  

为了进一步协助您查询订单状态、物流信息或处理其他问题，请您提供以下信息：  
1. **订单相关商品**（例如商品名称或类别）  
2. **需要协助的具体事项**（例如：查询物流、修改地址、退换货等）  

我会根据您的需求，尽快为您提供帮助！ 😊
================================ Human Message =================================

我的订单号是多少？
================================== Ai Message ==================================

根据之前的对话记录，您提到的订单号是 **12345**。  

如果您需要查询此订单的物流、状态或其他信息，请随时告诉我，我会尽力协助您！ 📦✨


**客服系统**

In [11]:
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()


model = init_chat_model(
    model="deepseek:deepseek-chat",
    temperature=0.1,
    max_tokens=2000
)

@tool
def get_order_status(order_id: str) -> str:
    """查询订单状态"""
    orders = {
        "12345": "已发货，预计明天送达",
        "67890": "配送中，今天下午送达"
    }
    return orders.get(order_id, "订单不存在")
    

# 客户今天上午咨询    
with SqliteSaver.from_conn_string("D:/GP/sqdata/checkpoints.sqlite") as checkpointer:
    agent = create_agent(model=model, tools=[get_order_status], checkpointer=checkpointer)

    config = {"configurable": {"thread_id": "zhang"}}
    agent.invoke({"messages": [{"role": "user", "content": "订单 12345 在哪？"}]}, config)


# 下午客户再次咨询（即使服务重启）
with SqliteSaver.from_conn_string("D:/GP/sqdata/checkpoints.sqlite") as checkpointer:
    agent = create_agent(model=model, tools=[get_order_status], checkpointer=checkpointer)
    response = agent.invoke({"messages": [{"role": "user", "content": "到了吗？"}]}, config)  
    # Agent 记得上午查询的订单号！
for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

订单 12345 在哪？
================================== Ai Message ==================================

我目前无法直接查询订单的具体位置信息。我主要能帮您查询订单的状态，比如是否已发货、配送进度等。

如果您想了解订单 12345 的当前状态，我可以帮您查询。您需要我查看这个订单的状态吗？
================================ Human Message =================================

到了吗？
================================== Ai Message ==================================

我来帮您查询订单 12345 的当前状态。
Tool Calls:
  get_order_status (call_00_ZzjfnQkxtvG1lbMud2fDrlFp)
 Call ID: call_00_ZzjfnQkxtvG1lbMud2fDrlFp
  Args:
    order_id: 12345
================================= Tool Message =================================
Name: get_order_status

已发货，预计明天送达
================================== Ai Message ==================================

根据查询结果，订单 12345 已经发货了，预计明天送达。所以您的订单还没有到，但明天应该就能收到了。
================================ Human Message =================================

订单 12345 在哪？
================================== Ai Message ==========

**多用户会话管理**

In [12]:
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()


model = init_chat_model(
    model="deepseek:deepseek-chat",
    temperature=0.1,
    max_tokens=2000
)

db_path = "multi_user.sqlite"
with SqliteSaver.from_conn_string(db_path) as checkpointer:
    agent = create_agent(
        model=model,
        tools=[],
        system_prompt="你是一个有帮助的助手。",
        checkpointer=checkpointer
    )

    # 用户 A
    print("\n[用户 A 的对话]")
    config_a = {"configurable": {"thread_id": "user_alice"}}
    agent.invoke(
        {"messages": [{"role": "user", "content": "我是 Alice，我喜欢编程"}]},
        config_a
    )
    print("Alice: 我是 Alice，我喜欢编程")

    # 用户 B
    print("\n[用户 B 的对话]")
    config_b = {"configurable": {"thread_id": "user_bob"}}
    agent.invoke(
        {"messages": [{"role": "user", "content": "我是 Bob，我喜欢设计"}]},
        config_b
    )
    print("Bob: 我是 Bob，我喜欢设计")

    # 回到用户 A
    print("\n[用户 A 继续对话]")
    response_a = agent.invoke(
        {"messages": [{"role": "user", "content": "我喜欢什么？"}]},
        config_a
    )
    print(f"Alice: 我喜欢什么？")
    print(f"Agent: {response_a['messages'][-1].content}")

    # 回到用户 B
    print("\n[用户 B 继续对话]")
    response_b = agent.invoke(
        {"messages": [{"role": "user", "content": "我喜欢什么？"}]},
        config_b
    )
    print(f"Bob: 我喜欢什么？")
    print(f"Agent: {response_b['messages'][-1].content}")


[用户 A 的对话]
Alice: 我是 Alice，我喜欢编程

[用户 B 的对话]
Bob: 我是 Bob，我喜欢设计

[用户 A 继续对话]
Alice: 我喜欢什么？
Agent: 你好 Alice！根据我们之前的对话，你明确提到过**喜欢编程**。不过，如果你是在问更具体的兴趣方向（比如编程语言、项目类型），或者**编程之外的其他爱好**（比如音乐、运动、阅读等），可能需要你多分享一些信息哦～  

如果暂时没有具体方向，也可以说说你最近在做什么，或者对什么感到好奇？ 😊

[用户 B 继续对话]
Bob: 我喜欢什么？
Agent: Bob，从对话中我能明确知道的是：**你喜欢设计**。但“设计”本身是一个很广阔的领域，所以这个问题可能是在引导你向内探索自己的偏好细节。  

或许可以试着问自己：  
- **“我在设计时，做什么最投入、最容易忘记时间？”**  
  （比如画草图、配色、建模、研究用户需求？）  
- **“我被哪种设计作品吸引？”**  
  （例如科技感界面、手工陶瓷、城市海报、可持续产品…）  
- **“如果抛开外界评价，我最想设计什么？”**  

有时候，喜欢的东西会藏在你的行为里：  
👉 你收藏的设计网站、反复看的作品集、自发做的练习项目…都可能暗示答案。  

需要的话，我可以陪你一步步梳理～ （或者你直接告诉我一个具体方向，我们深入聊聊！）🎨


**带工具的持久化 Agent**

In [16]:
db_path = "tools.sqlite"

with SqliteSaver.from_conn_string(db_path) as checkpointer:
    agent = create_agent(
        model=model,
        tools=[get_order_status],
        system_prompt="你是一个有帮助的助手。",
        checkpointer=checkpointer
    )

    config = {"configurable": {"thread_id": "customer_001"}}

    print("\n第一轮：查询订单")
    print("客户: 查询订单 12345 的状态")
    response1 = agent.invoke(
        {"messages": [{"role": "user", "content": "查询订单 12345 的状态"}]},
        config=config
    )
    print(f"Agent: {response1['messages'][-1].content}")

    print("\n第二轮：询问之前的查询结果")
    print("客户: 我的订单什么时候到？")
    response2 = agent.invoke(
        {"messages": [{"role": "user", "content": "我的订单什么时候到？"}]},
        config=config
    )
    print(f"Agent: {response2['messages'][-1].content}")


第一轮：查询订单
客户: 查询订单 12345 的状态
Agent: 订单 12345 的状态是：**已发货，预计明天送达**。

第二轮：询问之前的查询结果
客户: 我的订单什么时候到？
Agent: 根据查询结果，您的订单 12345 已经发货，预计**明天**送达。如果您需要更具体的送达时间，建议您联系物流公司或查看订单详情页面获取更精确的物流跟踪信息。
